# 拼接式蒸馏与图像生成式蒸馏

本章我们继续来说数据集蒸馏的方法，其中 RDED 是一种将原始真实数据集进行拼接的蒸馏方法，D4M 则是一种利用先验分布进行蒸馏的方法。我们详细说。

# RDED

RDED 是 Realistic, Diverse, and Efficient Dataset Distillation 的缩写。原文是 https://arxiv.org/pdf/2312.03526 On the Diversity and Realism of Distilled Dataset: An Efficient Dataset Distillation Paradigm。

## 基本逻辑

我们认为蒸馏数据集必须包含两种重要属性：真实性与高的表达信息密度。前者是说，合成数据集应当接近真实图像，不包含仅对某个网络有效的奇怪噪声，这通常意味着强大的跨架构泛化能力。其次是，一张图最好同时包含多种类内子模式与信息，这有利于提升低 IPC 下蒸馏性能。

然而，以上两种属性实际上存在张力或者矛盾。过度追求自然图像，可能只是选出几张普通真实图片，信息压缩能力有限；过度追求信息密度，则可能生成 MTT 那种人眼难以理解且依赖特定网络的纹理图案。我们想方设法尽力保留二者。

RDED 基本思想是，从原始真实数据中裁剪并筛选真实 patch，再将多个选中的 patch 拼接成新的蒸馏图像。这意味着真实性来自 patch 本身直接取自的真实图像，多样性来自不同 patch 取自的不同原始样本，信息密度来自一张蒸馏图里塞入的多个 patch。更多的，这是一种 Optimization-free 的方案，我指完全不需要任何梯度下降优化。

RDED 方法相较 MTT 这类 Bilevel optimization-based distillation 方法的好处是完全抛弃了繁重的双层梯度优化，这使得计算效率大大提升；而 RDED 相较 SRe2L 这类 Uni-level optimization-based distillation 方法的好处是，SRe2L 需要一个专家模型可以完全体现原始真实数据集信息的假设，而实际上这个假设往往不成立，这意味着专家模型其实仅仅表现出了部分真实数据集的内容。

而 RDED 最直接的上游方法是 CoreSet selection-based distillation，如 Random 和 Herding，我们还没怎么介绍过这些方法。核心思想是直接选择真实样本，缺点是单张样本信息密度低以及极低 IPC 下性能瓶颈。RDED 可以视为将这些方法改造为最小单位为 patch 的挑选。

我们正式开始叙述。设真实数据集是
$$\mathcal T
=
(\widetilde X,\widetilde Y)
=
\{(\widetilde x_i,\widetilde y_i)\}_{i=1}^{|\mathcal T|},$$
蒸馏数据集是
$$\mathcal S
=
(X,Y)
=
\{(x_j,y_j)\}_{j=1}^{|\mathcal S|}, \quad |\mathcal S|
\ll
|\mathcal T|.$$
理想目标是
$$\sup_{(x,y)\sim\mathcal T}
\left|
\ell(\phi_{\theta_{\mathcal T}}(x),y)
-
\ell(\phi_{\theta_{\mathcal S}}(x),y)
\right|
\lt
\epsilon.$$

我们考虑一族 Observer models 是 $\mathcal V$，这里的模型可以是完全不同架构不同容量的模型。一个高质量的蒸馏数据集应该覆盖广泛的特征与场景，其次是应该被多种架构 Observer models 正确理解的，我们暂且称这种模型族层面的可识别性为真实性。最后的，这种蒸馏算法必须能够处理各类大规模数据集与高分辨率数据集，这是我们的效率要求。

因此我们引入一个概念 $\mathcal{V}$ - information，这个量可以非常粗糙地估计多样性。定义是
$$I_{\mathcal V}(X\to Y)
=
H_{\mathcal V}(Y\mid\varnothing)
-
H_{\mathcal V}(Y\mid X).$$
其中 $X$ 是输入图像，$Y$ 是输入标签，$H_{\mathcal V}(Y\mid\varnothing)$ 表示 Observer 不看图像时预测标签的最小交叉熵，$H_{\mathcal V}(Y\mid X)$ 表示 Observer 看到图像后预测标签的最小交叉熵。

请注意，$H_{\mathcal V}(Y\mid X)$ 是一个比较容易理解的量。如果一个模型可以完全准确地判断每张图片所属类别，这一项直接是零。但是 $H_{\mathcal V}(Y\mid\varnothing)$ 比较陌生，这指的是假如我们仅仅输出一个常数 logits，对所有图像计算交叉熵损失平均最小值
$$H_{\mathcal V}(Y\mid\varnothing)=
\inf_{f\in\mathcal V}
\mathbb E_{y\sim Y}
\left[
-\log f[\varnothing](y)
\right].$$
而 $H_{\mathcal V}(Y\mid X)$ 指的是
$$H_{\mathcal V}(Y\mid X)=
\inf_{f\in\mathcal V}
\mathbb E_{(x,y)\sim(X,Y)}
\left[
-\log f[x](y)
\right].$$
这意味着 $I_{\mathcal V}(X\to Y)$ 实际上衡量了一个数据集的多样性与可预测性。如果数据集本身分布很狭窄，这个差会很小。但是这个估计实际上非常粗糙，RDED 方法最终也没有直接优化这个量。我们可以将其视为一种理论包装。

最后我们希望
$$\mathcal S
=
\arg\max_{(X,Y)\in\mathcal A(\mathcal T)}
I_{\mathcal V}(X\to Y)$$
其中 $\mathcal A(\mathcal T)$ 是允许由真实数据构造出的候选蒸馏数据集集合。

我需要再次提醒，从 RDED 整个方法来看 $\mathcal{V}$ - information 仅仅起到原则上的指导作用。接下来我们来叙述真正的工程方法。

实际工程中，我们不可能采样全部的 Observer 模型。我们将其简化为
$$\mathcal V
=
\{\phi_h,\phi_{\theta_{\mathcal T}}\},$$
其中 $\phi_h$ 表示人类标注者，$\phi_{\theta_{\mathcal T}}$ 表示一个在真实数据集上预训练的 Observer 模型。

现在我们定义 Realism 分数。对于一个 patch $\xi_{i,k}$
$$s_{i,k}
=
-
\ell
\left(
\phi_{\theta_{\mathcal T}}(\xi_{i,k}),
\widetilde y_i
\right),$$
其中 $\widetilde y_i$ 是原始真实图像标签，$\ell$ 是分类损失，如交叉熵。所以 Realism 分数实际上就是 Observer 对于该 patch 置信程度。

记第 $i$ 个数据集元素为
$$(\widetilde x_i,\widetilde y_i).$$
我们随机裁剪出 $K$ 个 patch
$$\{\xi_{i,k}\}_{k=1}^{K}.$$
然后每个 patch resize 到原始图片尺寸并且计算
$$s_{i,k}
=
-
\ell
\left(
\phi_{\theta_{\mathcal T}}(\xi_{i,k}),
\widetilde y_i
\right).$$
最后我们选择一个 Realism 分数最高的 patch
$$\xi_{i,*}
=
\arg\max_{\xi_{i,k}}
s_{i,k}.$$
所以我们选择了一个最能够代表该图片特征的 patch。

接下来我们筛选一个类内的蒸馏数据集。定义
$$\mathcal T_c
=
\{(\widetilde x_i,\widetilde y_i)\in\mathcal T
\mid
\widetilde y_i=c\}.$$
这个类内集合太大，还可能导致 selection bias。因此我们实际上仅仅选择一个随机子集
$$\mathcal T_c'
\subset
\mathcal T_c,$$
这保证了一定程度多样性。对于这个子集，每张图得到一个 key patch 之后，记候选池
$$\mathcal Q_c
=
\{
(\xi_{i,*},s_{i,*})
\mid
\widetilde x_i\in\mathcal T_c
\}.$$
然后我们挑选分数最高的 $N\times\mathrm{IPC}$ 个 patch 得到 $\mathcal Q_c'$。这里 $N$ 表示每张图片由多少张 patch 拼成。

现在我们从 $\mathcal Q_c'$ 中不放回地取出 $N$ 个随机 patch
$$\{\xi_{i,*}\}_{i=1}^{N}
\subset
\mathcal Q_c'.$$
然后拼接
$$x_j
=
\operatorname{concatenate}
\left(
\{\xi_{i,*}\}_{i=1}^{N}
\right).$$
比如 $N = 4$，实际上一张蒸馏图片就是
$$\begin{matrix}
\text{patch}_1  \quad\text{patch}_2\\
\text{patch}_3  \quad\text{patch}_4
\end{matrix}.$$
最终我们会得到 IPC 张类内蒸馏图片，这正是我们要的。

对于这些蒸馏图片，我们还需要给出一个 Soft Label。这里的标签非常有趣，我们不是为整张图打一个标签而是为各个区域给出标签。

我们将拼接图 $x_j$ 进行多次随机裁剪得到多个区域，对于区域 $x_{j,m}$
$$q_{j,m}
=
\operatorname{softmax}
\left(
\phi_{\theta_{\mathcal T}}(x_{j,m})
\right),$$
所以最终标签是
$$x_j
\longrightarrow
\{q_{j,1},q_{j,2},\ldots,q_{j,N}\}.$$
这不是传统的 logits 标签，因为学生模型学习也不是传统的学习。学生模型使用合成数据集进行训练时，不做传统交叉熵而是各个区域交叉熵平均
$$\mathcal L
=
-
\sum_j
\sum_{m=1}^{N}
q_{j,m}^{\mathsf T}
\log
\phi_{\theta_{\mathcal S}}
(x_{j,m}).$$
其中 $\phi_{\theta_{\mathcal S}}(x_{j,m})$ 是学生模型预测概率，这是一个逐 patch 的学习。所以一种理解是，RDED 方法实际上把一个蒸馏数据集总数量放大了 $N$ 倍但是每张图片分辨率下降 $\sqrt{N}$ 倍。

请注意，裁剪的区域不一定是原始被拼接的 patch。

下面这张图展示了完整的 RDED 方法。

<img src="./assets/RDED.png" width="900" height="300">

## 蒸馏算法

下面展示了完整的算法。

$$
\begin{array}{l}
\hline
\textbf{Algorithm 1 } \text{RDED: An efficient framework for high-} \\
\text{resolution dataset distillation  } \\
\hline
\textbf{Input: } \text{Original full dataset } \mathcal{T}, \text{ a corresponding pre-} \\
\text{trained observer model } \phi_{\boldsymbol{\theta}_{\mathcal{T}}} \text{ and initial } \mathcal{S} = \varnothing. \\
\begin{aligned}
& \textbf{for } \mathcal{T}_c' \subset \mathcal{T}_c \subset \mathcal{T} \textbf{ do} \\
& \quad \textbf{for } (\hat{\mathbf{x}}_i, \hat{y}_i) \in \mathcal{T}_c' \textbf{ do} \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \ \ \triangleright \color{#FF69B4}{\text{Stage 1}} \\
& \quad\quad \text{Crop } \hat{\mathbf{x}}_i \text{ into } K \text{ patches } \{\xi_{i,k}\}_{k=1}^K \\
& \quad\quad \textbf{for } k = 1 \textbf{ to } K \textbf{ do} \\
& \quad\quad\quad \text{Calculate the score } s_{i,k} = -\ell(\phi_{\boldsymbol{\theta}_{\mathcal{T}}}(\xi_{i,k}), \hat{y}_i) \\
& \quad\quad \textbf{for end} \\
& \quad\quad \text{Select patch } \xi_{i,\star} \text{ from } \{\xi_{i,k}\}_{k=1}^K \text{ via } s_{i,\star} \\
& \quad \textbf{for end} \\
& \quad \text{Select top-}(N \times \text{IPC}) \text{ patches} \\
& \quad \textbf{for } j = 1 \textbf{ to } \text{IPC} \textbf{ do} \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \ \ \triangleright \color{#FF69B4}{\text{Stage 2}} \\
& \quad\quad \text{Squeeze } N \text{ selected patches into } \mathbf{x}_j \\
& \quad\quad \text{Relabel } \mathbf{x}_j \text{ with } y_j \\
& \quad\quad \mathcal{S} = \mathcal{S} \cup \{(\mathbf{x}_j, y_j)\} \\
& \quad \textbf{for end} \\
& \textbf{for end}
\end{aligned} \\
\textbf{Output: } \text{Small distilled dataset } \mathcal{S} \\
\hline
\end{array}
$$

我们讲一些细节。首先 $\mathcal{T}_c'$ 大概需要多大？原作者选取了 $300$ 张
$$\mathcal T_c'
=
\{\widetilde x_i\}_{i=1}^{300}.$$
其次的，每张图片裁剪出多少张 patch？原作者将 $K$ 设置为 $5$
$$\{\xi_{i,k}\}_{k=1}^{5},$$
所以最终我们得到
$$\{(\xi_{i,*},s_{i,*})\}_{i=1}^{300}.$$

# RDED 的总结

RDED 达到了其承诺的高真实性与高效率与高多样性，这使得其成为一种出色的数据集蒸馏方法。

RDED 的缺陷是，低分辨率时效率低于 MTT 与 DATM 等等方法。这是因为低分辨率时我们难以将一张合成数据分割成多个 patch，这可以视为一种对于计算量的代偿。但是在高分辨率时，IPC 越大 RDED 相较于 TESLA 等等方法优势越大。同时，其计算量与巅峰显存远远小于传统方法。这展现了其在高分辨率图像上非常强劲的性能。

接下来我想说说 D4M。本教程原意是图像生成的基本原理教程，但是我们本专题似乎一直与图像生成没什么关系。这个刻板印象应该在这里被打破，我们引入一种基于 Diffusion 模型的数据集蒸馏，这正是我们无比熟悉的领域。

# D4M

原文是 https://arxiv.org/pdf/2407.15138 $D^4 M$: Dataset Distillation via Disentangled Diffusion Model 同样为了简称我们将其记为 D4M。